# Pipeline ML

Ce notebook ajoute au projet Kedro le pipeline `training`, qui optimise un LightGBM, et le pipeline `loading`, qui lit les CSV depuis Cloud Storage.
Les trois pipelines sont ensuite enchaînés dans un pipeline `global`.
Le formatage PEP 8 est appliqué avant publication, avec flake8, black et pre-commit.
Séparer `processing` et `training` permet de relancer un seul des deux.

## Pipeline `training`

Créer `src/purchase_predict/pipelines/training/` avec `__init__.py`, `nodes.py` et `pipeline.py`.
`__init__.py` est le même que pour `processing`.

`nodes.py` contient trois fonctions.
`train_model` instancie un estimateur et l'ajuste.
`optimize_hyp` cherche des hyperparamètres avec Hyperopt et une validation `RepeatedKFold` (4 plis, 1 répétition).
`auto_ml` est la fonction du node : elle retient le meilleur modèle et le retourne sous la clé `model`.

Hyperopt minimise. La métrique passée est `-f1_score`, donc la recherche maximise le F1.
Les hyperparamètres de `override_schemas` sont convertis en entiers avant `fit`.
Hyperopt peut produire des flottants (`10.0`) là où LightGBM exige un `int` (`max_depth`, `num_leaves`, `min_child_samples`, `num_iterations`).

Déclarer le modèle dans `conf/base/catalog.yml`.
Le type `pickle.PickleDataset` sérialise l'estimateur dans `data/06_models/model.pkl`.

Ajouter `automl_max_evals` à `conf/base/parameters.yml`.

Installer les dépendances, puis lancer le pipeline.
Avec 4 plis et `automl_max_evals: 10`, la recherche ajuste 40 modèles. Compter 3 à 5 minutes.
Après une modification de pipeline, relancer `kedro viz`.

## Pipeline `loading`

Le pipeline `processing` suppose que `primary.csv` existe déjà.
`loading` le construit à partir des CSV Spark déposés dans Cloud Storage.
Créer `src/purchase_predict/pipelines/loading/` avec les trois mêmes fichiers.

`load_csv_from_bucket` télécharge les objets dont le nom se termine par `.csv` sous le préfixe `.../part-`, puis les concatène.
Le client `storage.Client()` utilise les identifiants de l'environnement.

Ajouter les deux paramètres à `conf/base/parameters.yml`.
`gcp_project_id` est l'identifiant du projet Google Cloud.
`gcs_primary_folder` est le dossier des CSV à encoder. Le premier segment est le bucket.

L'exécution sans identifiant échoue : `Anonymous caller does not have storage.objects.list access`.
Créer un compte de service nommé `purchase-predict`, avec le rôle Lecteur des objets de l'espace de stockage.
Le principe du moindre privilège s'applique : ce rôle couvre la lecture, et rien d'autre.
La description du compte indique quelles applications l'utilisent.

La clé JSON équivaut à un mot de passe.
La placer dans `conf/local/service-account.json`. Ce dossier est ignoré par Git.
La variable `GOOGLE_APPLICATION_CREDENTIALS` est à redéfinir dans chaque nouveau terminal.

Le pipeline écrit `primary.csv`.
Trois pipelines sont alors disponibles.

- `loading` construit `primary.csv` depuis le bucket.
- `processing` encode ce fichier et le découpe.
- `training` ajuste le modèle.

Un changement d'encodeur se relance avec `processing`. Un modèle supplémentaire se relance avec `training`.
Le pipeline `global` enchaîne les trois, dans cet ordre.
`training` dure plus longtemps sur l'historique complet que sur l'échantillon.

Kedro 0.17 appelle `register_pipelines()` sans argument.
La signature ci-dessous est celle du cours, paramètre `self` compris.
Si la fonction est définie au niveau du module `src/purchase_predict/pipeline_registry.py`, ce paramètre provoque une `TypeError` : le retirer dans ce cas.
Le module importe `Pipeline`, `Dict`, et les paquets `processing`, `training` et `loading` sous les noms utilisés ci-dessous.

## Linting du projet

Les fichiers Python du projet suivent PEP 8.
`flake8` signale les écarts. `black` reformate le code.
Les deux outils utilisent la même longueur de ligne : 120 caractères.
La complexité maximale retenue pour flake8 est 16.

Fichier `.flake8` à la racine.
Section Black à ajouter dans `pyproject.toml`, déjà présent à la racine. Black lit `[tool.black]` dans ce fichier.

Exécuter `flake8` à la racine, puis `black`, puis `flake8` à nouveau.
La seconde passe de flake8 ne doit rien afficher.

`pre-commit` lance black puis flake8 à chaque commit.
Le commit local n'est accepté que si les deux crochets passent.
Le fichier `.pre-commit-config.yaml` se place à la racine.
`pre-commit install` enregistre les crochets. L'initialisation prend quelques dizaines de secondes.

Pousser ensuite vers le dépôt distant.
Dans chaque nouveau terminal, l'agent SSH doit être relancé et la clé privée ajoutée avant `git push`.